# Live debugging: Core AI values in your `forward`

`coreai_torch.debugging.live` runs a model **eagerly** while every tensor it produces carries the value the **converted Core AI program** computed for it, as `tensor.coreai`. Set a breakpoint anywhere in `forward` with your IDE, stop there, and compare the two side by side — in the Variables pane, a watch expression, or the debug console.

**To use it in VS Code**

1. Select this repository's `.venv` as the kernel, and start from a fresh kernel.
2. Run the cells down to and including *Inputs* normally.
3. Click in the gutter of the **model cell** to set breakpoints — inside `Attention.forward`, say, on `weights = torch.softmax(scores, dim=-1)`.
4. Run the **Debug** cell with **Debug Cell**.
5. When it stops, expand a tensor such as `scores` in the Variables pane and open its `coreai` attribute, or evaluate `scores.coreai` in the debug console. Continue to the next breakpoint.

The model runs on the **interpreter**, which fuses nothing, so most lines have a value of their own. That needs the Core AI runtime bundled with the wheel, which the next cell selects — it has to run before anything imports `coreai`.

In [ ]:
# Before anything imports coreai: the runtime is chosen at that import.
import os
import sys

os.environ["USE_LOCAL_COREAI"] = "1"
# Both set means neither: the runtime falls back to its default.
os.environ.pop("USE_OS_COREAI", None)

loaded = sys.modules.get("coreai.runtime")
if loaded is not None and getattr(loaded, "_use_os_coreai", False):
    raise RuntimeError(
        "this kernel already imported coreai with the OS framework, so "
        "USE_LOCAL_COREAI is read too late to matter. Restart the kernel and "
        "run this cell first."
    )

import coreai.runtime  # noqa: E402

print(
    "Core AI runtime:", "OS framework" if coreai.runtime._use_os_coreai else "bundled"
)

## The model

A pre-norm transformer block, with two submodules — `Attention` and `MLP` — each behind a `LayerNorm`. It lives in this cell rather than a file so you can put breakpoints on it here.

In [ ]:
import math

import torch
from torch import nn
from torch.nn import functional as F


class Attention(nn.Module):
    def __init__(self, dim: int, heads: int) -> None:
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, 3 * dim)
        self.out = nn.Linear(dim, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch, seq, dim = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(dim, dim=-1)
        q = q.view(batch, seq, self.heads, -1).transpose(1, 2)
        k = k.view(batch, seq, self.heads, -1).transpose(1, 2)
        v = v.view(batch, seq, self.heads, -1).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        weights = torch.softmax(scores, dim=-1)
        context = weights @ v
        merged = context.transpose(1, 2).reshape(batch, seq, dim)
        return self.out(merged)


class MLP(nn.Module):
    def __init__(self, dim: int, hidden: int) -> None:
        super().__init__()
        self.up = nn.Linear(dim, hidden)
        self.down = nn.Linear(hidden, dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.up(x)
        a = F.gelu(h)
        return self.down(a)


class Block(nn.Module):
    def __init__(self, dim: int = 16, heads: int = 2, hidden: int = 32) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim, hidden)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normed = self.norm1(x)
        attended = self.attn(normed)
        x = x + attended
        normed = self.norm2(x)
        fed = self.mlp(normed)
        y = x + fed
        return y

## Inputs

In [ ]:
torch.manual_seed(0)
model = Block().eval()
x = torch.randn(1, 4, 16)
print(model)

## Debug

Set your breakpoints in the model cell above, then run this cell with **Debug Cell**. It exports and converts the model, loads the program on the interpreter, and runs `forward` eagerly — stopping at your breakpoints, where every tensor in scope has its `coreai` value. Run it normally to skip the stops.

In [ ]:
from coreai_torch.debugging.live import live

with live(model) as session:
    with torch.no_grad():
        y = model(x)

print("y.coreai:", y.coreai)

## Every tensor, eager against Core AI

`session.report()` lists each tensor the run produced, in call order, with how far the Core AI value is from the eager one — or why there is none.

In [ ]:
def show(rows):
    for row in rows:
        if row["match"] is None:
            verdict = "unobservable — " + row["note"][:90]
        else:
            verdict = f"{'✓' if row['match'] else '✗ MISMATCH'}  max|Δ| {row['max_abs_diff']:.1e}"
        print(f"{row['call']:14} {row['node']:11} {str(row['shape']):14} {verdict}")
    matched = sum(bool(row["match"]) for row in rows)
    wrong = sum(row["match"] is False for row in rows)
    print(f"\n{matched} / {len(rows)} match, {wrong} mismatch")


show(session.report())

## Reading the unobservable rows

A value is never guessed: when the program does not expose it unambiguously, the row says why instead.

- **The two `LayerNorm`s** lower to a `layer_norm` composite, which the interpreter inlines. Their values appear once Core AI maps each call through that inlining (rdar://189232142); until the pinned Core AI includes it, they are unobservable.
- **The three pieces of `qkv.split`** have one shape and dtype, and the bundled interpreter delivers a multi-output op's results in no fixed order — so which piece is which cannot be told.
- **A few ops the converter elides** — a `permute` folded into a later transpose, the reshapes around a batched matmul — have no Core AI op to read from.

Everything else is the program's own value. To try a variation, edit the model cell, re-run it and the *Inputs* cell, and debug again.